![Astrofisica Computacional](../../../new_logo.png)

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Hvar Observatory\
    Faculty of Geodesy, University of Zagreb\
    Zagreb, Croatia

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & jcamposro@geof.hr)

---

# NumPy

Ejercicios avanzados de NumPy + Funciones/Clases + Archivos

**Conjunto de datos:** Adaptado del conjunto de datos de temperaturas de Helsinki de 2017 (https://raw.githubusercontent.com/csmastersUH/data_analysis_with_python_2020/master/kumpula-weather-2017.csv)

Incluye:
- NumPy: segmentación (*slicing*), máscaras, ajuste polinómico (*polyfit*), FFT.

- Funciones/Clases: herencia, decoradores, métodos vectorizados.

- Archivos: `loadtxt`, `savetxt` con datos procesados.

In [9]:
import numpy as np

## Ejercicio 1: Carga de datos + Preprocesamiento con NumPy

Carga las temperaturas desde el archivo "kumpula-weather-2017.csv". Escribe una función para convertir el formato mm-dd en días transcurridos desde el inicio (days = monthday_to_days).

- Filtra los valores NaN, inf o masks.

- Calcula las anomalías (temperatura - media móvil de 30 días).

- Detecta valores atípicos (método del IQR: Q1 - 1.5 * IQR).

**Resultado esperado:** `temps_clean` (N x 2), `days` (N,).

In [11]:

#Cargar archivos desde un .csv
data = np.genfromtxt('kumpula-weather-2017.csv', delimiter=',', names=True)

#Función para convertir mm-dd a días transcurridos
def monthday_to_days(data):
    # Intentamos extraer las columnas de mes y día. 
    try:
        meses = data['m']
        dias = data['d']
    except ValueError:
        raise ValueError("No se encontraron las columnas 'm' y 'd' en el CSV.")

    # Lista de días por mes para el año 2017
    #0 al principio para alinear los índices (Enero = 1)
    dias_por_mes = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    
    # np.cumsum calcula la suma acumulada. 
    dias_acumulados = np.cumsum(dias_por_mes)
    
    # Calculamos el día del año: 
    # Días acumulados de los meses anteriores + el día actual del mes
    # Usamos (meses - 1) como índice para buscar en el arreglo
    total_days = dias_acumulados[meses.astype(int) - 1] + dias
    
    return total_days

days = monthday_to_days(data)
print(days)


[  1.   2.   3.   4.   5.   6.   7.   8.   9.  10.  11.  12.  13.  14.
  15.  16.  17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.  28.
  29.  30.  31.  32.  33.  34.  35.  36.  37.  38.  39.  40.  41.  42.
  43.  44.  45.  46.  47.  48.  49.  50.  51.  52.  53.  54.  55.  56.
  57.  58.  59.  60.  61.  62.  63.  64.  65.  66.  67.  68.  69.  70.
  71.  72.  73.  74.  75.  76.  77.  78.  79.  80.  81.  82.  83.  84.
  85.  86.  87.  88.  89.  90.  91.  92.  93.  94.  95.  96.  97.  98.
  99. 100. 101. 102. 103. 104. 105. 106. 107. 108. 109. 110. 111. 112.
 113. 114. 115. 116. 117. 118. 119. 120. 121. 122. 123. 124. 125. 126.
 127. 128. 129. 130. 131. 132. 133. 134. 135. 136. 137. 138. 139. 140.
 141. 142. 143. 144. 145. 146. 147. 148. 149. 150. 151. 152. 153. 154.
 155. 156. 157. 158. 159. 160. 161. 162. 163. 164. 165. 166. 167. 168.
 169. 170. 171. 172. 173. 174. 175. 176. 177. 178. 179. 180. 181. 182.
 183. 184. 185. 186. 187. 188. 189. 190. 191. 192. 193. 194. 195. 196.
 197. 

## Ejercicio 2: Clase avanzada con herencia y decoradores

Crea una **clase base `TimeSeriesAnalyzer`**:
- `smooth(self, window=7)`: media móvil.

- Decorador `@vectorize` para aplicar funciones a las columnas.

La **clase derivada `WeatherAnalyzer`** hereda y añade:
- `seasonal_decompose(self)`: tendencia (ajuste polinómico de grado 2), componente estacional (FFT, 4 frecuencias principales) y residuo.

- `forecast(self, days_ahead=30)`: pronóstico simple tipo ARIMA (últimos 30 días, ajuste polinómico de grado 3) + ruido.

- **Uso:** `analyzer = WeatherAnalyzer(days, temps_clean[:,1])`

In [12]:
# Decorator to vectorize method over the columns
def vectorize(method):
    def wrapper(self, *args, **kwargs):
        results = np.array([method(self, col, *args, **kwargs) for col in self.data.T]).T
        return results if results.ndim > 1 else results.flatten()
    return wrapper

class TimeSeriesAnalyzer:
    """Clase base para análisis de series temporales con NumPy."""
    def __init__(self, t: np.ndarray, data: np.ndarray):
        self.t = np.asarray(t)
        self.data = np.asarray(data)
        if self.t.shape[0] != self.data.shape[0]:
            raise ValueError("t y data deben tener misma longitud")
    
    @vectorize
    def smooth(self, col: np.ndarray, window: int = 7, mode: str = 'same') -> np.ndarray:
        """Media móvil con convolve."""
        kernel = np.ones(window) / window
        return np.convolve(col, kernel, mode=mode)
    
    def stats(self) -> dict:
        """Estadísticas básicas por columna."""
        return {
            'mean': np.mean(self.data, axis=0),
            'std': np.std(self.data, axis=0),
            'min': np.min(self.data, axis=0),
            'max': np.max(self.data, axis=0)
        }

class WeatherAnalyzer(TimeSeriesAnalyzer):
    """Hija especializada en datos meteorológicos."""
    def __init__(self, t: np.ndarray, data: np.ndarray):
        super().__init__(t, data)
    
    def seasonal_decompose(self, degree_trend: int = 2, n_freqs: int = 4) -> dict:
        """Descomposición: trend (polyfit), seasonal (FFT top freqs), residual."""
        # Trend: polyfit global
        p_trend = np.polynomial.Polynomial.fit(self.t, self.data, degree_trend)
        trend = p_trend(self.t)
        
        # Seasonal: FFT, top n_freqs armónicos
        fft = np.fft.fft(self.data - trend, axis=0)
        freqs = np.fft.fftfreq(len(self.t))
        top_idx = np.argsort(np.abs(fft), axis=0)[-n_freqs:][::-1]
        seasonal = np.zeros_like(self.data)
        for idx in top_idx[:]:
            seasonal[:] += 2 * np.real(np.fft.ifft(fft[:] * (np.abs(fft[idx]) > 1e-3)))
        
        residual = self.data - trend - seasonal
        return {'trend': trend, 'seasonal': seasonal, 'residual': residual}
    
    def forecast(self, days_ahead: int = 30, degree: int = 3, noise_std: float = 1.0) -> tuple:
        """Pronóstico simple: polyfit últimos datos + ruido gaussiano."""
        n_last = min(degree * 10, len(self.t) // 2)
        t_last = self.t[-n_last:]
        data_last = self.data[-n_last:]
        
        t_future = np.linspace(self.t[-1], self.t[-1] + days_ahead, days_ahead)
        forecast = np.zeros(days_ahead) #np.zeros((days_ahead, self.data.shape[1]))
        
        p = np.polyfit(t_last, data_last[:], degree)
        forecast[:] = np.polyval(p, t_future) + np.random.normal(0, noise_std, days_ahead)
        
        return t_future, forecast

# # DEMO de USO (¡ejecuta después de Ej.1!)
#analyzer = WeatherAnalyzer(days, temps_clean)
#smoothed = analyzer.smooth(window=15)
#decomp = analyzer.seasonal_decompose()
#t_fc, fc = analyzer.forecast(30)
#print(analyzer.stats())

## Ejercicio 3: I/O robusta + datos procesados

- Guarda `temps_clean` y `anomalies` en el archivo 'processed_weather.npz' (usando `np.savez`).

- Guarda un subconjunto (primeros 100 días; columnas: `days`, `temp_smooth`, `anomaly`) en 'subset.csv' (usando `savetxt`, con `fmt='%.2f'` y encabezado).

- Función `load_and_validate(filename)`: carga el archivo npz/csv, verifica las dimensiones y la presencia de valores infinitos o NaN, y devuelve un diccionario.

In [ ]:
# np.savez('processed_weather.npz', temps=temps_clean, anomalies=anomalies)
# subset = np.column_stack([dias[:100], smoothed[0,:100], anomalies[:100]])
# np.savetxt('subset.csv', subset, delimiter=',', header='day,temp_smooth,anomaly', fmt='%.3f')

def load_and_validate(filename):
    # Maneja .npz (load), .csv; check np.isfinite.all(), shape==(?,3)
    pass

# Algunas preguntas:

- Para los valores atípicos (método del IQR): ¿qué fracción de los datos se elimina? ¿Es un enfoque conservador o agresivo? Proponga una alternativa (p. ej., sigma=3).
- Analice la ejecución de `@vectorize`: para `data.shape=(365,2)`, ¿cuántas llamadas se realizan a `smooth(col)`? ¿Por qué `results.T`? ¿Qué otro método de vectorización propone?
- En `seasonal_decompose()`: ¿por qué se resta la tendencia antes de la FFT? ¿Qué representan las 4 frecuencias principales (diaria/semanal/mensual/anual)?
- Ejecute `analyzer.stats()` comparando los datos originales frente a los suavizados: ¿cuál es el porcentaje de reducción de la desviación estándar por columna? ¿Por qué `polyfit` de grado 2 captura bien la tendencia?